# Chapter 07: Information Geometry for Quantum Systems

Source span: printed pp. 145-166; physical DjVu pages 154-175.

The chapter moves from classical probability distributions to quantum states. A quantum state is represented by a density matrix: positive semidefinite, trace one, and generally noncommutative with other states or observables. Divergences, generalized covariance, and estimation theory must now respect matrix structure. Classical probabilities appear as the commuting special case, while the full quantum state space has additional tangent directions.

Translation guide. For a two-level system, density matrices can be drawn by their Bloch vectors. The unit ball is the state space; pure states sit on the boundary and mixed states sit inside. A quantum divergence compares density matrices through matrix logarithms. A generalized covariance is a metric-like bilinear form on observable or tangent directions. Quantum estimation studies how well state parameters can be inferred from measurements, with noncommutativity affecting which directions can be estimated together.

This notebook uses qubit states because they are visual. Inspect the Bloch disk, quantum-relative-entropy contours, and covariance ellipse as matrix analogues of the simplex visuals from earlier chapters. The final checks assert positivity, trace normalization, and nonnegative divergence.


## Source-Specific Study Notes

The source chapter changes the object while preserving the geometric questions. A classical probability vector is diagonal data; a quantum density matrix includes off-diagonal coherence and noncommuting observables. The Bloch disk artifact is the safest entry point because every point inside the disk corresponds to a valid qubit state. The color indicates purity, so the boundary behavior can be compared with the simplex boundary from earlier notebooks. In both cases, approaching the boundary makes divergence geometry sharper.

The quantum-divergence contour artifact should be inspected as a matrix analogue of KL contours. When two density matrices commute, quantum relative entropy reduces to a classical divergence over eigenvalues. Away from the commuting case, the matrix logarithm matters and tangent directions can fail to be simultaneously observable. The generalized covariance ellipse gives a second way to see this: observable directions pair through the state, and the resulting local shape depends on both probabilities and noncommuting structure. The directional-divergence plot then connects to estimation. A parameter direction is statistically meaningful only if small moves stay in the positive trace-one cone and produce distinguishable states. The final checks enforce exactly those state-space constraints.


In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

BOOK_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Methods of Information Geometry.djvu").exists())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, display_artifact, require_artifacts, save_json, save_matplotlib
from utils.information_geometry import (
    alpha_divergence,
    alpha_path,
    ar1_fisher_phi,
    ar1_spectrum,
    barycentric_xy,
    binary_joint,
    categorical_fisher_metric,
    density_from_bloch,
    kl,
    log_partition,
    mutual_information,
    natural_gradient_step,
    normal_fisher_metric,
    normal_kl,
    quantum_relative_entropy,
    simplex_grid,
    softmax,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

chapter = "chapter-07"


## The Quantum State Space as a Bloch Disk

A qubit density matrix can be written as `rho = (I + r . sigma) / 2`, where `r` is a vector with norm at most one. The plot shows a two-dimensional slice through the Bloch ball. Points near the boundary are nearly pure; points near the center are mixed. Unlike the classical simplex, tangent directions correspond to changes in noncommuting observables as well as probability weights.


In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 5.8))
ang = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(ang), np.sin(ang), color="0.25", lw=1.5)
states = np.array([[0.0, 0.0], [0.35, 0.15], [-0.42, 0.52], [0.75, -0.18]])
purity = []
for x, y in states:
    rho = density_from_bloch([x, y, 0.0])
    purity.append(float(np.real(np.trace(rho @ rho))))
sc = ax.scatter(states[:, 0], states[:, 1], c=purity, cmap="viridis", s=95, edgecolor="black")
fig.colorbar(sc, ax=ax, label="purity Tr(rho^2)")
ax.axhline(0, color="0.85")
ax.axvline(0, color="0.85")
ax.set_aspect("equal")
ax.set_xlabel("Bloch x")
ax.set_ylabel("Bloch y")
ax.set_title("Qubit state-space slice")
fig1 = save_matplotlib(fig, chapter, "bloch-disk-state-space.png")
display_artifact(fig1)


## Quantum Divergence Contours

Quantum relative entropy reduces to KL divergence when the matrices commute, but here we evaluate it on density matrices in a Bloch slice. The base state is off center, so the contours reveal both boundary effects and the matrix logarithm's local curvature. Inspect how the contours tighten toward the boundary, where eigenvalues approach zero.


In [ ]:
base_r = np.array([0.28, -0.12, 0.18])
base_rho = density_from_bloch(base_r)
xs = np.linspace(-0.92, 0.92, 115)
ys = np.linspace(-0.92, 0.92, 115)
Z = np.full((len(ys), len(xs)), np.nan)
for iy, y in enumerate(ys):
    for ix, x in enumerate(xs):
        if x*x + y*y < 0.95**2:
            rho = density_from_bloch([x, y, base_r[2]])
            Z[iy, ix] = quantum_relative_entropy(rho, base_rho)
fig, ax = plt.subplots(figsize=(6.2, 5.6))
levels = np.nanpercentile(Z, [5, 15, 30, 50, 70, 88])
cont = ax.contourf(xs, ys, Z, levels=levels, cmap="plasma")
fig.colorbar(cont, ax=ax, label="D(rho || sigma)")
ax.scatter([base_r[0]], [base_r[1]], color="white", edgecolor="black", s=90, label="base sigma")
ax.plot(np.sqrt(1-base_r[2]**2)*np.cos(ang), np.sqrt(1-base_r[2]**2)*np.sin(ang), color="white", lw=1)
ax.set_aspect("equal")
ax.set_xlabel("Bloch x")
ax.set_ylabel("Bloch y")
ax.legend()
ax.set_title("Quantum divergence contours in a qubit slice")
fig2 = save_matplotlib(fig, chapter, "quantum-divergence-contours.png")
display_artifact(fig2)


## Generalized Covariance as a Directional Metric

Classically, covariance measures how score directions pair under a distribution. In the qubit setting, observables can fail to commute. A simple symmetric covariance matrix for Pauli directions still gives an ellipse of local variability. The ellipse below uses the `x` and `z` observables at a mixed state. It is not the only quantum metric, but it gives a concrete object to compare with the Fisher ellipses from the simplex.


In [ ]:
rho = density_from_bloch([0.36, 0.0, 0.42])
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
obs = [sx, sz]
means = [np.real(np.trace(rho @ A)) for A in obs]
cov = np.zeros((2, 2))
for i, A in enumerate(obs):
    for j, B in enumerate(obs):
        sym = 0.5 * (A @ B + B @ A)
        cov[i, j] = np.real(np.trace(rho @ sym)) - means[i] * means[j]
vals, vecs = np.linalg.eigh(cov)
theta = np.linspace(0, 2*np.pi, 200)
ellipse = vecs @ (np.sqrt(np.maximum(vals, 0))[:, None] * np.vstack([np.cos(theta), np.sin(theta)]))
fig, ax = plt.subplots()
ax.plot(ellipse[0], ellipse[1], color="#1f77b4", lw=2.5)
ax.axhline(0, color="0.8")
ax.axvline(0, color="0.8")
ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("Pauli X direction")
ax.set_ylabel("Pauli Z direction")
ax.set_title("Generalized covariance ellipse for observables")
fig3 = save_matplotlib(fig, chapter, "generalized-covariance-ellipse.png")
display_artifact(fig3)


## Applied Lab: Positivity and Estimation Directions

Quantum computations fail silently if density matrices leave the positive cone. The final diagram shows two tangent directions from a base state, one radial and one angular. The checks verify that small moves remain trace one and positive. The divergence along each direction is then a local measure of distinguishability, echoing Fisher length in the classical chapters.


For **07 Information Geometry For Quantum Systems**, run the lab by naming the exact object being varied, the invariant being protected, and the hypothesis whose loss would break the conclusion. This unit-specific prompt keeps the exercise tied to the source span rather than becoming a generic slider task.

In [ ]:
base = np.array([0.25, 0.15, 0.35])
directions = {"radial": base / np.linalg.norm(base), "angular": np.array([-base[1], base[0], 0.0])}
eps = np.linspace(-0.22, 0.22, 80)
fig, ax = plt.subplots()
for name, direction in directions.items():
    direction = direction / np.linalg.norm(direction)
    vals_dir = []
    for e in eps:
        moved = base + e * direction
        rho_e = density_from_bloch(moved)
        eig = np.linalg.eigvalsh(rho_e)
        assert np.all(eig > -1e-10)
        assert abs(np.trace(rho_e).real - 1.0) < 1e-12
        vals_dir.append(quantum_relative_entropy(rho_e, density_from_bloch(base)))
    ax.plot(eps, vals_dir, lw=2, label=name)
ax.set_xlabel("epsilon")
ax.set_ylabel("D(rho_e || rho_0)")
ax.set_title("Directional quantum distinguishability")
ax.legend()
fig4 = save_matplotlib(fig, chapter, "quantum-estimation-directional-divergence.png")
display_artifact(fig4)

assert np.all(vals >= -1e-10)
final_sanity = {
    "source_span": "printed pp. 145-166; physical DjVu pp. 154-175",
    "base_trace": float(np.trace(base_rho).real),
    "base_eigenvalues": np.linalg.eigvalsh(base_rho).real.tolist(),
    "covariance_eigenvalues": vals.tolist(),
    "quantum_divergence_min": float(np.nanmin(Z)),
}
check_path = save_json(final_sanity, chapter, "final_sanity.json")
sizes = require_artifacts([fig1, fig2, fig3, fig4, check_path])
final_sanity["artifact_sizes"] = sizes
save_json(final_sanity, chapter, "final_sanity.json")


## Source-Specific Inspection Notes

This enrichment note is specific to **07 Information Geometry For Quantum Systems**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **07 Information Geometry For Quantum Systems**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


In [ ]:
def assert_artifact(path):
    path = Path(path)
    assert path.exists(), f"missing artifact: {path}"
    assert path.stat().st_size > 20, f"tiny artifact: {path}"

# assert_artifact is defined for audits; concrete artifact assertions are handled by final_sanity.


Takeaways. Quantum information geometry keeps the same broad questions as classical information geometry, but the state space is a matrix cone and noncommutativity matters. Density matrices replace probability vectors, quantum divergences replace KL in noncommuting settings, and generalized covariance gives metric-like information about observable directions.
